# Downsampling Benchmark

Purpose: run controlled target-cell downsampling and compare RNA-only PCA, protein-only PCA, and joint RNA+protein PCA.

Inputs: `data/processed/pbmc5k_10x_citeseq_representations.h5ad` and `config/benchmark_config.yaml`.

Outputs: raw benchmark table, metric summary, target-count grid, minimal controls, and run summary.

Matching script: `scripts/run_benchmark.py`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "rarecell").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import anndata as ad
import yaml
from rarecell.benchmark import INTERMEDIATE_DIR, LOGS_DIR, METRICS_DIR, TABLES_DIR, cell_counts_by_fraction, \
    make_metric_summary, run_downsampling_benchmark, validate_results_table
from rarecell.downsampling import summarize_downsampling_grid
from rarecell.utils import make_output_prefix, resolve_label_key, resolve_target_label, write_json


In [ ]:
CONFIG = PROJECT_ROOT / "config" / "benchmark_config.yaml"
config = yaml.safe_load(CONFIG.read_text())
INPUT = PROJECT_ROOT / config["dataset_path"]
adata = ad.read_h5ad(INPUT)

label_key = resolve_label_key(adata, preferred=config.get("label_column"))
target_label = resolve_target_label(adata, label_key, preferred=config.get("target_cell_type"))
representations = config.get("representations", ["rna_pca", "protein_pca", "joint_pca"])
fractions = [float(v) for v in config.get("fractions", [1.0, 0.5, 0.25, 0.1, 0.05])]
seeds = [int(v) for v in config.get("seeds", [0, 1, 2, 3, 4])]
n_neighbors = int(config.get("k_neighbors", 15))
output_prefix = make_output_prefix(config.get("dataset", "citeseq"), target_label)


In [ ]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

grid = summarize_downsampling_grid(adata, label_key, target_label, fractions, seeds)
grid.to_csv(TABLES_DIR / f"{output_prefix}__downsampling_grid.csv", index=False)

results = run_downsampling_benchmark(
    adata,
    label_key=label_key,
    target_label=target_label,
    representation_keys=representations,
    retain_fractions=fractions,
    seeds=seeds,
    n_neighbors=n_neighbors,
    dataset=config.get("dataset", "citeseq"),
)
validate_results_table(results, representations, fractions, seeds)
results.to_csv(TABLES_DIR / f"{output_prefix}__benchmark_results.csv", index=False)
results.to_csv(METRICS_DIR / f"{output_prefix}__benchmark_raw.csv", index=False)


In [ ]:
metric_summary = make_metric_summary(results)
metric_summary.to_csv(TABLES_DIR / f"{output_prefix}__metric_summary.csv", index=False)
counts = cell_counts_by_fraction(adata, label_key, target_label, fractions, seeds)
counts.to_csv(INTERMEDIATE_DIR / f"{output_prefix}__cell_counts_by_fraction.csv", index=False)

run_summary = {
    "input_file": str(INPUT.relative_to(PROJECT_ROOT)),
    "label_key": label_key,
    "target_label": str(target_label),
    "output_prefix": output_prefix,
    "representations": representations,
    "retain_fractions": fractions,
    "seeds": seeds,
    "n_neighbors": n_neighbors,
    "n_cells": int(adata.n_obs),
    "target_cell_count_original": int((adata.obs[label_key].astype(str) == str(target_label)).sum()),
    "n_cell_types": int(adata.obs[label_key].astype(str).nunique()),
    "output_files": [
        f"results/tables/{output_prefix}__benchmark_results.csv",
        f"results/tables/{output_prefix}__metric_summary.csv",
        f"results/metrics/{output_prefix}__benchmark_raw.csv",
    ],
}
write_json(run_summary, LOGS_DIR / f"{output_prefix}__run_summary.json")
metric_summary.head()
